# Custom Surface Site Analysis : Part 1 
## Jupyter notebook for identifying uniques sites in defect(s)

Edit **Cell 2 only**, then **Run All**.


In [ ]:
# Cell 1: import
import sys, os
sys.path.append("/home/shikim/pynta")
from ase import Atom, Atoms
import copy
from ase.io import read, write, Trajectory
from acat.settings import CustomSurface
from acat.adsorption_sites import SlabAdsorptionSites
from ase.build import surface,fcc111
from ase.visualize import view
import site_analysis as sa

In [ ]:
# Cell 2: inputs
xyz_path = "/home/shikim/pynta/pynta/preprocessing/custom_surfaces/defects/pt111_defect_middle.xyz"
n_layers=4
adsorbate_height = 1.0
site_bond_cutoff = 1.5
tag_symbol = "Ne"     # used only if defect is detected
dz = 0.1
stable_steps = 3
verbose = True


In [ ]:
#visualize
slab = read(xyz_path)
view(slab, viewer='x3d')

NameError: name 'get_site' is not defined

In [ ]:
# Cell 3: Defect detection workflow -
# By running this cell, you will learn:
# 1. Number of defect site(s)
# 2. The coordinate of the center of the void 
# 3. Number of identified sites
# 4. Files created
#       a. traj file for geometries that determines unique sites on the surface
#       b. traj file for geometries that determines unique sites inside of the void
#       c. xyz file of the geometry inside of the void with maximum coordination number
#       d. All site lists in json file (geom_all_sites_lists.json)
#       e. Neighborlist in json file (neighbor_site_list.json) 
sa.workflow_auto(
    xyz_path=xyz_path,
    adsorbate_height=adsorbate_height,
    site_bond_cutoff=site_bond_cutoff,
    tag_symbol=tag_symbol,
    dz=dz,
   # surface_string_no_defect = surface_string_no_defect,
    stable_steps=stable_steps,
    verbose=verbose,
)

In [ ]:

# ── Graph construction & comparison ──────────────────────────────────────────
import json, numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import Counter
from ase.io import read

# ── 1. Load data written by workflow_auto ─────────────────────────────────────
# Adjust the filenames if your workflow used different paths.
try:
    geoms = read("geom_all.traj", index=":")        # defect workflow
    sites_json_path = "geom_all_sites_lists.json"
except Exception:
    geoms = read("unique_sites.traj", index=":")    # no-defect workflow
    sites_json_path = "sites_graph.json"

with open(sites_json_path) as f:
    raw = json.load(f)

# Convert to list-of-lists format expected by classify_all_sites
# raw is either [{geom_index, sites:[...]}, ...] or [{site:..., morphology:...}, ...]
if isinstance(raw, list) and raw and "sites" in raw[0]:
    single_sites_lists = [entry["sites"] for entry in raw]
else:
    single_sites_lists = [[s] for s in raw]

single_geoms = geoms

# ── 2. Run graph pipeline ─────────────────────────────────────────────────────
admols, geom_indices = sa.classify_all_sites(single_geoms, single_sites_lists)
iso_mat, clusters    = sa.cluster_isomorphic_graphs(admols)
geom_to_label, key_to_label = sa.update_site_labels_by_graph_and_type(
    single_sites_lists, clusters, geom_indices
)

# reverse map: label → (cluster_id, original_site, morphology)
label_to_key   = {v: k for k, v in key_to_label.items()}
labels_sorted  = sorted(label_to_key, key=lambda x: int(x[4:]))

# one representative admol per label
label_to_admol = {}
for graph_idx, geom_idx in enumerate(geom_indices):
    label = geom_to_label.get(geom_idx)
    if label and label not in label_to_admol:
        label_to_admol[label] = admols[graph_idx]

# ── helper: RMG Molecule → networkx Graph ─────────────────────────────────────
def rmg_to_nx(mol):
    G = nx.Graph()
    atom_to_idx = {atom: i for i, atom in enumerate(mol.atoms)}
    for i, atom in enumerate(mol.atoms):
        sym = atom.element.symbol if hasattr(atom.element, "symbol") else str(atom.element)
        G.add_node(i, element=sym)
    for i, atom in enumerate(mol.atoms):
        for other in atom.edges:
            j = atom_to_idx[other]
            if i < j:
                G.add_edge(i, j)
    return G

# ── 3. Figure 1: one graph per label ─────────────────────────────────────────
ELEM_COLOR = {
    "X":  "#aaaaaa",   # surface metal (generic)
    "Ne": "#00bfff",   # adsorbate tag (Ne mapped in RMG)
    "Li": "#ffd700",   # He mapped to Li in RMG
    "C":  "#404040", "O": "#e04040",
    "N":  "#4040e0", "H": "#eeeeee",
}

def node_colors(G):
    return [ELEM_COLOR.get(G.nodes[n]["element"], "#cccccc") for n in G.nodes()]

n      = len(labels_sorted)
ncols  = min(5, n)
nrows  = (n + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 4 * nrows), squeeze=False)
axes_flat = axes.flatten()

for ax, label in zip(axes_flat, labels_sorted):
    mol = label_to_admol[label]
    G   = rmg_to_nx(mol)
    pos = nx.kamada_kawai_layout(G)

    nx.draw_networkx_edges(G, pos, ax=ax, edge_color="#666666", width=1.5, alpha=0.7)
    nx.draw_networkx_nodes(G, pos, ax=ax, node_color=node_colors(G),
                           node_size=300, edgecolors="#333333", linewidths=0.8)
    nx.draw_networkx_labels(G, pos, ax=ax,
                            labels={nd: G.nodes[nd]["element"] for nd in G.nodes()},
                            font_size=6)

    _, orig_site, morph = label_to_key[label]
    ax.set_title(f"{label}\n{orig_site} | {morph}", fontsize=8)
    ax.axis("off")

for ax in axes_flat[n:]:
    ax.set_visible(False)

legend_patches = [mpatches.Patch(color=c, label=e) for e, c in ELEM_COLOR.items()]
fig.legend(handles=legend_patches, loc="lower center", ncol=len(ELEM_COLOR),
           fontsize=8, frameon=False, bbox_to_anchor=(0.5, -0.01))
plt.suptitle(
    "One graph per unique site label\n"
    "(same label  ⟺  isomorphic graph  +  same site type  +  same morphology)",
    fontsize=11
)
plt.tight_layout()
plt.show()

# ── 4. Figure 2: isomorphism matrix ──────────────────────────────────────────
n_g = len(admols)
fig2, ax2 = plt.subplots(figsize=(max(5, n_g * 0.28), max(4, n_g * 0.28)))
im = ax2.imshow(iso_mat.astype(float), cmap="Blues", vmin=0, vmax=1, aspect="auto")
ax2.set_title("Graph isomorphism matrix  (dark = isomorphic pair)", fontsize=11)
ax2.set_xlabel("Site index")
ax2.set_ylabel("Site index")
plt.colorbar(im, ax=ax2, shrink=0.7, label="isomorphic (1=yes)")
plt.tight_layout()
plt.show()

# ── 5. Summary table ──────────────────────────────────────────────────────────
label_counts = Counter(
    s["site"]
    for sites in single_sites_lists
    for s in sites
    if str(s.get("site", "")).startswith("site")
)

print(f"Total sites identified   : {len(single_sites_lists)}")
print(f"Distinct graph clusters  : {len(clusters)}")
print(f"Distinct site labels     : {len(key_to_label)}")
print()
print(f"{'Label':<10} {'Count':>6}  {'Original site':<14} Morphology")
print("─" * 52)
for label in labels_sorted:
    _, site_val, morph_val = label_to_key[label]
    print(f"{label:<10} {label_counts[label]:>6}  {site_val:<14} {morph_val}")